In [ ]:
# ===================================================================================
# GROUP MIGRATION SCRIPT
# Migrates Portal Groups from Source (10.9.1) to Target (11.3).
# - Creates groups with matching properties (title, tags, description, snippet, access).
# - Copies thumbnails.
# - Adds tracking tag "SourceGroupID:{id}" for traceability.
# ===================================================================================

import pandas as pd
import json
import copy
import time
import csv
import os
import datetime
import urllib3
import requests
import sys
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =============================================================================
# --- CONFIGURATION (from shared config) ---------------------------------------
# =============================================================================
from migration_config import *

# --- ORCHESTRATOR SIDECAR LOADER ---
_sidecar_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_sidecar_7_groups.json")
if os.path.exists(_sidecar_path):
    import json as _json
    GROUP_IDS_TO_MIGRATE = _json.load(open(_sidecar_path))["ids"]
    print(f"[Orchestrator] Loaded {len(GROUP_IDS_TO_MIGRATE)} Group IDs from sidecar.")
else:
    GROUP_IDS_TO_MIGRATE = [
        # Example Source Group IDs
        # "a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4",
    ]

# =============================================================================
# --- LOGGING & CONNECT --------------------------------------------------------
# =============================================================================
if not os.path.exists(TEMP_DIR):
    os.makedirs(TEMP_DIR)

STATS = {
    "groups_scanned": 0,
    "groups_created": 0,
    "groups_skipped_exists": 0,
    "groups_skipped_log": 0,
    "failures": 0,
}

ALREADY_MIGRATED_IDS = set()
if os.path.exists(LOG_FILE):
    try:
        df_log = pd.read_csv(LOG_FILE)
        if "SourceID" in df_log.columns:
            ALREADY_MIGRATED_IDS = set(df_log["SourceID"].astype(str).str.strip())
        print(f"Loaded history: {len(ALREADY_MIGRATED_IDS)} items previously migrated.")
    except:
        pass
else:
    try:
        with open(LOG_FILE, mode="w", newline="") as f:
            csv.writer(f).writerow(["SourceID", "LayerIndex", "TargetID", "Title", "MigratedDate", "Type"])
    except:
        pass


def log_migration(source_id, target_id, title):
    try:
        with open(LOG_FILE, mode="a", newline="") as f:
            csv.writer(f).writerow([
                source_id, "", target_id, title,
                datetime.datetime.now().strftime("%m/%d/%Y %H:%M"), "Group"
            ])
    except:
        print("   ⚠ Log Write Failed")


print("Connecting...")
try:
    session = requests.Session()
    retry_strategy = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "POST"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    source_gis = GIS(url=SOURCE_URL, token=SOURCE_TOKEN, verify_cert=False, referer=SOURCE_URL, session=session)
    target_gis = GIS(url=TARGET_URL, token=TARGET_TOKEN, verify_cert=False, referer=TARGET_URL, session=session)

    if not target_gis.users.me:
        raise Exception("Target login failed.")
    print(f" > Source: {source_gis.users.me.username}")
    print(f" > Target: {target_gis.users.me.username}")

except Exception as e:
    print(f"❌ CONNECTION FAILURE: {e}")
    raise SystemExit(1)

# =============================================================================
# --- HELPER: COPY THUMBNAIL ---------------------------------------------------
# =============================================================================
def copy_thumbnail(source_group, target_group):
    """Download thumbnail from source group and upload to target group."""
    try:
        print("      🖼️ Copying Thumbnail...")
        temp_dir = os.path.join(TEMP_DIR, "thumbnails_temp")
        os.makedirs(temp_dir, exist_ok=True)
        thumb_path = source_group.download_thumbnail(save_folder=temp_dir)
        if thumb_path:
            target_group.update(thumbnail=thumb_path)
            print("         ✅ Thumbnail copied.")
        else:
            print("         - No thumbnail found on source group.")
    except Exception as e:
        print(f"      ⚠ Thumbnail copy failed: {e}")

# =============================================================================
# --- MAIN LOOP ----------------------------------------------------------------
# =============================================================================
print("\nStarting Group Migration...")
start_time = time.time()

for group_id in GROUP_IDS_TO_MIGRATE:
    try:
        STATS["groups_scanned"] += 1

        # CHECK 1: HISTORY LOG
        if str(group_id) in ALREADY_MIGRATED_IDS:
            print(f"\n[Skip] {group_id} found in migration log.")
            STATS["groups_skipped_log"] += 1
            continue

        # GET SOURCE GROUP
        source_group = source_gis.groups.get(group_id)
        if not source_group:
            print(f"❌ Source group {group_id} not found.")
            STATS["failures"] += 1
            continue

        print(f"\nProcessing Group: {source_group.title} ({group_id})...")

        # CHECK 2: DOES GROUP WITH SAME TITLE ALREADY EXIST ON TARGET?
        existing_groups = target_gis.groups.search(f"title:\"{source_group.title}\"")
        matched = next((g for g in existing_groups if g.title == source_group.title), None)
        if matched:
            print(f"   ⚠ Group '{source_group.title}' already exists in target (ID: {matched.id}). Skipping.")
            STATS["groups_skipped_exists"] += 1
            # Still log it so we don't re-check next time
            log_migration(group_id, matched.id, matched.title)
            ALREADY_MIGRATED_IDS.add(str(group_id))
            continue

        # BUILD GROUP PROPERTIES
        group_tags = list(source_group.tags) if source_group.tags else []
        group_tags.append(f"SourceGroupID:{group_id}")
        if "Migrated" not in group_tags:
            group_tags.append("Migrated")

        group_properties = {
            "title": source_group.title,
            "tags": ", ".join(group_tags),
            "description": source_group.description or "",
            "snippet": source_group.snippet or "",
            "access": source_group.access or "private",
            "is_invitation_only": getattr(source_group, "isInvitationOnly", True),
            "is_view_only": getattr(source_group, "isViewOnly", False),
        }

        print(f"   Creating group '{source_group.title}' on target...")
        new_group = target_gis.groups.create_from_dict(group_properties)

        if not new_group:
            print(f"   ❌ Failed to create group '{source_group.title}'.")
            STATS["failures"] += 1
            continue

        print(f"   ✅ Group created: {new_group.id}")
        STATS["groups_created"] += 1

        # COPY THUMBNAIL
        copy_thumbnail(source_group, new_group)

        # LOG MIGRATION
        log_migration(group_id, new_group.id, new_group.title)
        ALREADY_MIGRATED_IDS.add(str(group_id))

        print(f"   💤 Cooling down for {THROTTLE_SECONDS} seconds...")
        time.sleep(THROTTLE_SECONDS)

    except Exception as e:
        print(f"❌ FAILED on group {group_id}: {e}")
        STATS["failures"] += 1

# =============================================================================
# --- REPORT -------------------------------------------------------------------
# =============================================================================
end_time = time.time()
total_seconds = int(end_time - start_time)
duration_str = f"{total_seconds // 3600}h {(total_seconds % 3600) // 60}m {total_seconds % 60}s"

print("\n" + "=" * 60)
print("        GROUP MIGRATION REPORT")
print("=" * 60)
print(f" ⏱️  Total Duration:             {duration_str}")
print("-" * 60)
print(f" 📦 Groups Scanned:            {STATS['groups_scanned']}")
print(f" 🧾 Skipped (log):             {STATS['groups_skipped_log']}")
print(f" ⏭️  Skipped (exists):           {STATS['groups_skipped_exists']}")
print(f" 🚀 Groups Created:            {STATS['groups_created']}")
print(f" ❌ Failures:                  {STATS['failures']}")
print("=" * 60)